# 1: Download Dependencies

In [1]:
pip install mediapipe

# 2: Import Libraries

In [ ]:
import os
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from glob import glob
from pathlib import Path
import shutil
import uuid
import kagglehub
from google.colab import drive
import shutil
import uuid
from tqdm import tqdm
import math
import traceback
import pandas as pd
from pathlib import Path
import uuid
import traceback
import os
import shutil

# 3: Get Datasets

In [3]:
synthetic_path = kagglehub.dataset_download("lexset/synthetic-asl-alphabet")
print("Path to synthetic dataset files:", synthetic_path)

Using Colab cache for faster access to the 'synthetic-asl-alphabet' dataset.
Path to synthetic dataset files: /kaggle/input/synthetic-asl-alphabet


# 4: Create directories

In [4]:
BASE_DIR = "/content/asl_project"

SOURCES = {
    "synthetic": synthetic_path
}

MERGED_ROOT = os.path.join(BASE_DIR, "dataset_root/merged")
LANDMARK_DIR = os.path.join(BASE_DIR, "dataset_root/landmarks")
CROP_DIR = os.path.join(BASE_DIR, "dataset_root/crops")
CSV_PATH = os.path.join(BASE_DIR, "dataset_root/metadata.csv")

for d in [MERGED_ROOT, LANDMARK_DIR, CROP_DIR]:
    os.makedirs(d, exist_ok=True)

# 5: PreProcess Images

## Set Configurations

In [5]:
IMG_EXTS = (".jpg", ".jpeg", ".png")
MP_MIN_CONF = 0.5
MP_MAX_HANDS = 2
MP_COMPLEXITY = 0
TRY_SCALES = (1.0,)
CROP_MARGIN = 0.25
SAVE_EVERY = 200
USE_SYMLINKS = True
mp_hands_module = mp.solutions.hands

## Increase Margin

In [6]:
def expand_bbox(xmin, ymin, xmax, ymax, w, h, margin=CROP_MARGIN):
    dw = (xmax - xmin) * margin
    dh = (ymax - ymin) * margin
    xmin = max(0, int(xmin - dw))
    ymin = max(0, int(ymin - dh))
    xmax = min(w, int(xmax + dw))
    ymax = min(h, int(ymax + dh))
    return xmin, ymin, xmax, ymax

## Crop and Extract Landmarks

In [7]:
def extract_landmarks_and_crop(img_path, save_id, mp_hands_obj,
                               crop_dir, landmark_dir,
                               try_scales=TRY_SCALES, crop_margin=CROP_MARGIN):
    img_path = Path(img_path)
    crop_dir = Path(crop_dir)
    landmark_dir = Path(landmark_dir)

    img = cv2.imread(str(img_path))
    if img is None:
        return {"status": "io_error"}

    h0, w0 = img.shape[:2]

    for scale in try_scales:
        if scale != 1.0:
            img_s = cv2.resize(img, (int(w0 * scale), int(h0 * scale)))
        else:
            img_s = img
        rgb = cv2.cvtColor(img_s, cv2.COLOR_BGR2RGB)
        result = mp_hands_obj.process(rgb)

        if result.multi_hand_landmarks:
            hand = result.multi_hand_landmarks[0]

            xs = [p.x for p in hand.landmark]
            ys = [p.y for p in hand.landmark]
            xmin, xmax = min(xs), max(xs)
            ymin, ymax = min(ys), max(ys)

            xmin_px, xmax_px = int(xmin * img_s.shape[1]), int(xmax * img_s.shape[1])
            ymin_px, ymax_px = int(ymin * img_s.shape[0]), int(ymax * img_s.shape[0])

            inv_scale = 1.0 / scale
            xmin_o, xmax_o = int(xmin_px * inv_scale), int(xmax_px * inv_scale)
            ymin_o, ymax_o = int(ymin_px * inv_scale), int(ymax_px * inv_scale)

            xmin_o, ymin_o, xmax_o, ymax_o = expand_bbox(xmin_o, ymin_o, xmax_o, ymax_o, w0, h0, margin=crop_margin)

            if xmin_o >= xmax_o or ymin_o >= ymax_o:
                continue

            crop = img[ymin_o:ymax_o, xmin_o:xmax_o]
            crop_path = str(crop_dir / f"{save_id}.jpg")
            cv2.imwrite(crop_path, crop)

            lmx = np.array([p.x * img_s.shape[1] * inv_scale for p in hand.landmark], dtype=np.float32)
            lmy = np.array([p.y * img_s.shape[0] * inv_scale for p in hand.landmark], dtype=np.float32)
            landmarks_pixels = np.stack([lmx, lmy], axis=1)
            landmarks_norm = landmarks_pixels / np.array([w0, h0], dtype=np.float32)

            lm_file = str(landmark_dir / f"{save_id}.npz")
            np.savez(lm_file, landmarks_pixels=landmarks_pixels, landmarks_norm=landmarks_norm)

            return {
                "status": "detected",
                "crop": crop_path,
                "landmark_file": lm_file,
                "bbox": (xmin_o, ymin_o, xmax_o, ymax_o)
            }

    return {"status": "no_detection"}

## Main processing funcion

In [ ]:
def process_all(sources_dict, csv_path=CSV_PATH, resume=True, save_every=SAVE_EVERY,
                merge_originals=True, only_missing=False):
    csv_p = Path(csv_path)
    merged_root_p = Path(MERGED_ROOT)
    merged_root_p.mkdir(parents=True, exist_ok=True)

    # Load existing CSV if resuming
    processed_paths = set()
    existing_df = None
    if csv_p.exists():
        try:
            existing_df = pd.read_csv(csv_p)
            if resume:
                processed_paths = set(existing_df['original_path'].astype(str).tolist())
        except Exception:
            existing_df = None
            processed_paths = set()

    cols = ["image_id", "source", "original_path", "label", "landmark_file", "crop_file"]
    buffer_rows = []
    stats = {"processed": 0, "detected": 0, "no_detection": 0, "io_error": 0, "merged": 0, "skipped": 0}

    # helper to flush buffer to CSV (append-safe)
    def flush_rows(rows):
        if not rows:
            return
        part_df = pd.DataFrame(rows, columns=cols)
        if csv_p.exists():
            part_df.to_csv(csv_p, mode='a', header=False, index=False)
        else:
            part_df.to_csv(csv_p, mode='w', header=True, index=False)

    mp_h = mp_hands_module.Hands(
        static_image_mode=True,
        max_num_hands=MP_MAX_HANDS,
        model_complexity=MP_COMPLEXITY,
        min_detection_confidence=MP_MIN_CONF
    )

    try:
        if only_missing:
            # Reprocess only rows missing crop/landmark
            if existing_df is None:
                print("No existing CSV to reprocess; nothing to do.")
                return stats
            mask = existing_df['crop_file'].isnull() | (existing_df['crop_file'].astype(str).str.strip() == "")
            rows_to_fix = existing_df[mask].to_dict(orient='records')
            for row in rows_to_fix:
                orig_path = Path(row['original_path'])
                if not orig_path.exists():
                    stats['io_error'] += 1
                    continue

                uid = row.get('image_id') if str(row.get('image_id','')).strip() else str(uuid.uuid4())
                label_norm = row.get('label', '').strip()
                if label_norm == 'Blank':
                    label_norm = 'Nothing'

                crop_subdir = Path(CROP_DIR) / label_norm
                landmark_subdir = Path(LANDMARK_DIR) / label_norm
                crop_subdir.mkdir(parents=True, exist_ok=True)
                landmark_subdir.mkdir(parents=True, exist_ok=True)


                meta = {
                    "image_id": uid,
                    "source": row.get('source', ''),
                    "original_path": str(orig_path.resolve()),
                    "label": label_norm,
                    "landmark_file": "",
                    "crop_file": ""
                }

                try:
                    res = extract_landmarks_and_crop(orig_path, uid, mp_h, str(crop_subdir), str(landmark_subdir))
                    if res.get("status") == "detected":
                        meta["landmark_file"] = res.get("landmark_file", "")
                        meta["crop_file"] = res.get("crop", "")
                        stats['detected'] += 1
                    elif res.get("status") == "io_error":
                        stats['io_error'] += 1
                    else:
                        stats['no_detection'] += 1
                except Exception:
                    traceback.print_exc()
                    stats['io_error'] += 1

                buffer_rows.append(meta)
                stats['processed'] += 1

                if len(buffer_rows) >= save_every:
                    flush_rows(buffer_rows)
                    buffer_rows[:] = []

        else:
            # Full walk of sources (recursive)
            for source_name, src_root in sources_dict.items():
                rootp = Path(src_root)
                if not rootp.exists():
                    print(f"Source not found: {rootp}")
                    continue

                imgs = [p for p in rootp.rglob("*") if p.suffix.lower() in IMG_EXTS]
                for img_path in tqdm(imgs, desc=f"Processing {source_name}"):
                    orig_str = str(img_path.resolve())
                    if orig_str in processed_paths:
                        stats['skipped'] += 1
                        continue

                    uid = str(uuid.uuid4())
                    # Extract label from the parent directory name
                    label = img_path.parent.name.strip()
                    label_norm = label
                    if label_norm == 'Blank':
                        label_norm = 'Nothing'

                    crop_subdir = Path(CROP_DIR) / label_norm
                    landmark_subdir = Path(LANDMARK_DIR) / label_norm
                    crop_subdir.mkdir(parents=True, exist_ok=True)
                    landmark_subdir.mkdir(parents=True, exist_ok=True)


                    meta = {
                        "image_id": uid,
                        "source": source_name,
                        "original_path": orig_str,
                        "label": label_norm,
                        "landmark_file": "",
                        "crop_file": ""
                    }

                    try:
                        res = extract_landmarks_and_crop(img_path, uid, mp_h, str(crop_subdir), str(landmark_subdir))
                        if res.get("status") == "detected":
                            meta["landmark_file"] = res.get("landmark_file", "")
                            meta["crop_file"] = res.get("crop", "")
                            stats['detected'] += 1
                        elif res.get("status") == "io_error":
                            stats['io_error'] += 1
                        else:
                            stats['no_detection'] += 1
                    except Exception:
                        traceback.print_exc()
                        stats['io_error'] += 1

                    buffer_rows.append(meta)
                    processed_paths.add(orig_str)
                    stats['processed'] += 1

                    # Merge original into MERGED_ROOT/<label> (symlink preferred)
                    if merge_originals:
                        dst_dir = merged_root_p / label_norm
                        dst_dir.mkdir(parents=True, exist_ok=True)
                        dst_path = dst_dir / f"{uid}_{img_path.name}"
                        try:
                            if USE_SYMLINKS:
                                if dst_path.exists():
                                    dst_path.unlink()
                                os.symlink(orig_str, str(dst_path))
                            else:
                                shutil.copy2(orig_str, str(dst_path))
                            stats['merged'] += 1
                        except Exception:
                            # fallback to copy
                            try:
                                shutil.copy2(orig_str, str(dst_path))
                                stats['merged'] += 1
                            except Exception:
                                pass

                    if len(buffer_rows) >= save_every:
                        flush_rows(buffer_rows)
                        buffer_rows[:] = []

        # final flush
        flush_rows(buffer_rows)
    finally:
        mp_h.close()

    return stats

process_all(SOURCES, csv_path=CSV_PATH, resume=True, save_every=200)

Processing synthetic: 100%|██████████| 27001/27001 [21:51<00:00, 20.59it/s]


{'processed': 27001,
 'detected': 24406,
 'no_detection': 2595,
 'io_error': 0,
 'merged': 27001,
 'skipped': 0}

In [9]:
!zip -r /content/asl_project/dataset_root/landmarks.zip /content/asl_project/dataset_root/landmarks
!zip -r /content/asl_project/dataset_root/crops.zip /content/asl_project/dataset_root/crops

Streaming output truncated to the last 5000 lines.
  adding: content/asl_project/dataset_root/crops/V/29ab7c01-ca45-40c1-90e4-63b2f76b81be.jpg (deflated 1%)
  adding: content/asl_project/dataset_root/crops/V/c6e7ecf0-0dcd-456a-b00a-eaf101f7dcf8.jpg (deflated 1%)
  adding: content/asl_project/dataset_root/crops/V/cb1705a9-ddaa-4436-ab7c-5f848993ad84.jpg (deflated 1%)
  adding: content/asl_project/dataset_root/crops/V/0a095ccb-c5aa-4023-bb0c-1676faa2bf23.jpg (deflated 1%)
  adding: content/asl_project/dataset_root/crops/V/563a4f9f-cdc3-423f-8ea2-e5919b582ac9.jpg (deflated 1%)
  adding: content/asl_project/dataset_root/crops/V/1470f5d2-2132-41b1-8ff3-fea5d01921e2.jpg (deflated 1%)
  adding: content/asl_project/dataset_root/crops/V/9071eaf8-e253-407a-b2b6-75f578b6b19a.jpg (deflated 1%)
  adding: content/asl_project/dataset_root/crops/V/03015009-b466-48d9-a321-3347339e5978.jpg (deflated 1%)
  adding: content/asl_project/dataset_root/crops/V/9aaf2ade-5dd0-41bd-8e0f-97fd91ec5530.jpg (deflated

In [11]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define source and destination paths
source_landmarks_zip = "/content/asl_project/dataset_root/landmarks.zip"
source_crops_zip = "/content/asl_project/dataset_root/crops.zip"
# destination_folder_id = "YOUR_FOLDER_ID_HERE" # Replace with your Google Drive folder ID

# Copy files to Google Drive
import shutil
import os

# Check if files exist before copying
if os.path.exists(source_landmarks_zip):
    shutil.copy(source_landmarks_zip, f"/content/drive/MyDrive/landmarks.zip")
    print("Uploaded landmarks.zip to Google Drive")
else:
    print(f"{source_landmarks_zip} not found.")

if os.path.exists(source_crops_zip):
    shutil.copy(source_crops_zip, f"/content/drive/MyDrive/crops.zip")
    print("Uploaded crops.zip to Google Drive")
else:
    print(f"{source_crops_zip} not found.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Uploaded landmarks.zip to Google Drive
Uploaded crops.zip to Google Drive
